# 🎵 Audio Sample Downloader

Streams audio samples from a Hugging Face dataset and saves them as files into `./audio_files/<language>/`.

**No FFmpeg / torchcodec required** — audio files are fetched directly from the HuggingFace Hub URL embedded in the `path` field.

**Transcript cleaning** — special Unicode/IPA characters are stripped via NFKD normalization.

In [3]:
import warnings
import logging
import os

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for name in ["datasets", "huggingface_hub", "transformers", "torchcodec", "torch"]:
    logging.getLogger(name).setLevel(logging.ERROR)

print("✅ Warning filters set")

✅ Warning filters set


In [4]:
import io
import re
import unicodedata
import requests
import numpy as np
import soundfile as sf

# Audio(decode=False) → returns raw {bytes, path} dict, skips torchcodec encode path
from datasets import load_dataset, Audio

print("✅ Imports OK")

✅ Imports OK


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  ← edit these lines
# ─────────────────────────────────────────────────────────────────────────────
DATASET_NAME  = "SilencioNetwork/amharic-speech"   # HuggingFace dataset repo
SPLIT_NAME    = "free_speech"                       # dataset split
LANGUAGE_NAME = "Amharic"                           # subfolder name under audio_files/
NUM_SAMPLES   = 3                                   # how many samples to download
OUTPUT_ROOT   = "C:/Users/gayat/Desktop/transcription_script/audio_files"                     # root output directory
HF_TOKEN      = None                               # optional: your HF token for private repos
# ─────────────────────────────────────────────────────────────────────────────

output_dir = os.path.join(OUTPUT_ROOT, LANGUAGE_NAME)
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Output directory: {os.path.abspath(output_dir)}")

✅ Output directory: C:\Users\gayat\Desktop\transcription_script\audio_files\Amharic


In [9]:
def fetch_audio_bytes(path: str, hf_token: str = None) -> bytes | None:
    """
    Fetches raw audio bytes from a HuggingFace Hub URL.

    WHY this is needed:
      This dataset (and many HF audio datasets) stores audio as separate files
      on HuggingFace Hub rather than embedding bytes inside the Parquet files.
      When you stream with Audio(decode=False), you get:
          bytes = None
          path  = 'hf://datasets/<repo>@<commit>/path/to/audio.wav'
      The path is a fully-qualified HF Hub URL, so we fetch it directly.
    """
    if not path:
        return None

    # Convert hf:// scheme to a real HTTPS URL
    # Format: hf://datasets/<repo>@<commit>/<relative_path>
    if path.startswith("hf://"):
        # Strip the hf:// prefix and split off the commit hash
        rest = path[len("hf://"):]
        # rest = "datasets/<repo>@<commit>/<rel_path>" or "datasets/<repo>/<rel_path>"
        if rest.startswith("datasets/"):
            rest = rest[len("datasets/"):]
        # Split at '@' to get repo and optional commit+path
        if "@" in rest:
            repo_and_before, after_at = rest.split("@", 1)
            # after_at = "<commit>/<rel_path>"
            commit, rel_path = after_at.split("/", 1) if "/" in after_at else (after_at, "")
            url = f"https://huggingface.co/datasets/{repo_and_before}/resolve/{commit}/{rel_path}"
        else:
            # No commit hash — use 'main'
            if "/" in rest:
                repo, rel_path = rest.split("/", 1)
                url = f"https://huggingface.co/datasets/{repo}/resolve/main/{rel_path}"
            else:
                return None
    elif path.startswith("http://") or path.startswith("https://"):
        url = path
    else:
        # Local path — try to read directly
        if os.path.isfile(path):
            with open(path, "rb") as f:
                return f.read()
        return None

    headers = {}
    if hf_token:
        headers["Authorization"] = f"Bearer {hf_token}"

    try:
        resp = requests.get(url, headers=headers, timeout=30)
        resp.raise_for_status()
        return resp.content
    except Exception as e:
        print(f"      [fetch error] {url}: {e}")
        return None


def save_audio_sample(audio_data, filepath: str, hf_token: str = None) -> bool:
    """
    Saves audio from a datasets sample to a .wav file.

    Handles all four shapes datasets can return:
      1. dict with 'bytes' (non-None)  → write raw bytes directly
      2. dict with 'bytes'=None + 'path' is hf:// URL → fetch then write
      3. dict with 'array' + 'sampling_rate' → re-encode as PCM-16 WAV
      4. raw bytes object → write directly
    """
    # Case 4: bare bytes (some old-format datasets)
    if isinstance(audio_data, (bytes, bytearray)):
        if len(audio_data) > 0:
            with open(filepath, "wb") as f:
                f.write(audio_data)
            return True
        return False

    if not isinstance(audio_data, dict):
        return False

    raw_bytes = audio_data.get("bytes")
    path      = audio_data.get("path")
    array     = audio_data.get("array")
    sr        = audio_data.get("sampling_rate")

    # Case 1: embedded bytes (dataset stores audio inline in parquet)
    if raw_bytes is not None and len(raw_bytes) > 0:
        with open(filepath, "wb") as f:
            f.write(raw_bytes)
        return True

    # Case 2: bytes is None, but path is a HF Hub URL → fetch it
    if path:
        fetched = fetch_audio_bytes(path, hf_token=hf_token)
        if fetched and len(fetched) > 0:
            with open(filepath, "wb") as f:
                f.write(fetched)
            return True

    # Case 3: decoded numpy array (shouldn't happen with decode=False, but just in case)
    if array is not None and sr is not None:
        arr = np.asarray(array, dtype=np.float32)
        sf.write(filepath, arr, int(sr), subtype="PCM_16")
        return True

    return False


def clean_text(text: str) -> str:
    """
    Strip IPA / extended-Latin Unicode characters from transcript text.
    Uses NFKD decomposition to separate base chars from diacritics, then
    drops the diacritics and any remaining non-ASCII characters.
    """
    if not isinstance(text, str):
        return text
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_only = "".join(ch for ch in nfkd if not unicodedata.category(ch).startswith("M"))
    ascii_str = ascii_only.encode("ascii", errors="ignore").decode("ascii")
    return re.sub(r" +", " ", ascii_str).strip()


print("✅ Helpers defined")

✅ Helpers defined


In [10]:
print(f"🚀 Streaming {NUM_SAMPLES} sample(s) from '{DATASET_NAME}' split='{SPLIT_NAME}'")
print(f"   → Saving to: {os.path.abspath(output_dir)}\n")

dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)

# Cast to Audio(decode=False) so datasets returns {bytes, path} without calling
# Audio.encode_example → which unconditionally imports torchcodec in datasets 3.x.
# bytes will be None for this dataset (audio stored as HF Hub files, not in parquet),
# so save_audio_sample fetches the file from the hf:// URL in the path field.
dataset = dataset.cast_column("audio", Audio(decode=False))

saved, skipped, failed = 0, 0, 0
TEXT_FIELDS = ("sentence", "transcription", "text", "transcript", "label")

for i, sample in enumerate(dataset.take(NUM_SAMPLES)):
    audio_data = sample.get("audio")

    if audio_data is None:
        for field in ("speech", "recording", "file"):
            audio_data = sample.get(field)
            if audio_data is not None:
                break

    if audio_data is None:
        print(f"  [{i+1}/{NUM_SAMPLES}] ⚠️  SKIP — no audio field. Keys: {list(sample.keys())}")
        skipped += 1
        continue

    if isinstance(audio_data, str):
        print(f"  [{i+1}/{NUM_SAMPLES}] ⚠️  SKIP — audio is a plain string path: {audio_data}")
        skipped += 1
        continue

    # Clean text transcript fields
    for tf in TEXT_FIELDS:
        if tf in sample and isinstance(sample[tf], str):
            original = sample[tf]
            cleaned  = clean_text(original)
            if cleaned != original:
                print(f"  [text/{tf}] {repr(original[:60])} → {repr(cleaned[:60])}")
            sample[tf] = cleaned

    # Derive a clean filename from the path hint
    raw_path = audio_data.get("path", "") if isinstance(audio_data, dict) else ""
    stem = os.path.splitext(os.path.basename(raw_path))[0] if raw_path else f"{LANGUAGE_NAME.lower()}_sample_{i+1}"
    out_file = os.path.join(output_dir, f"{stem}.wav")

    print(f"  [{i+1}/{NUM_SAMPLES}] {stem}.wav", end=" ... ", flush=True)

    try:
        ok = save_audio_sample(audio_data, out_file, hf_token=HF_TOKEN)
        if ok:
            size_kb = os.path.getsize(out_file) / 1024
            print(f"✅ saved ({size_kb:.1f} KB)")
            saved += 1
        else:
            print("⚠️  skipped (no usable payload)")
            skipped += 1
    except Exception as exc:
        print(f"❌ FAILED — {exc}")
        failed += 1

print(f"\n{'─'*38}")
print(f"✅ Saved   : {saved}")
print(f"⚠️  Skipped : {skipped}")
print(f"❌ Failed  : {failed}")
print(f"📂 Output  : {os.path.abspath(output_dir)}")

🚀 Streaming 3 sample(s) from 'SilencioNetwork/amharic-speech' split='free_speech'
   → Saving to: C:\Users\gayat\Desktop\transcription_script\audio_files\Amharic

  [1/3] audio_0bee2d5d-93b9-4262-9717-076a0b5d9318.wav ... ✅ saved (3937.5 KB)
  [2/3] audio_0c873bc9-289c-40f2-944b-66dfa82209d0.wav ... ✅ saved (4126.9 KB)
  [3/3] audio_0ce9a935-4603-4f76-bc05-b57473ace1b2.wav ... ✅ saved (3843.8 KB)

──────────────────────────────────────
✅ Saved   : 3
⚠️  Skipped : 0
❌ Failed  : 0
📂 Output  : C:\Users\gayat\Desktop\transcription_script\audio_files\Amharic
